In [ ]:
import os
%pwd

In [ ]:
# os.chdir('../')
%pwd

In [ ]:
from pathlib import Path
from dataclasses import dataclass

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir          : Path
    data_path         : Path
    tokenizer_path    : Path
    train_data_path   : Path
    test_data_path    : Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir          : Path
    model_path        : Path
    history_path      : Path

@dataclass(frozen=True)
class Params:
    max_words     : int
    max_len       : int
    test_size     : float
    random_state  : int
    batch_size    : int
    epochs        : int
    embedding_dim : int
    lstm_units    : int
    dropout_rate  : float
    learning_rate : float

In [ ]:
from src.IMDBSentimentAnalysis.constants import *
from src.IMDBSentimentAnalysis.utils.main import read_yaml, create_directories

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        create_directories([config.root_dir])

        return DataTransformationConfig(
            root_dir          = config.root_dir,
            data_path         = config.data_path,
            tokenizer_path    = config.tokenizer_path,
            train_data_path   = config.train_data_path,
            test_data_path    = config.test_data_path
        )
    
    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        create_directories([config.root_dir])
        return ModelTrainerConfig(
            root_dir          = config.root_dir,
            model_path        = config.model_path,
            history_path      = config.history_path
        )
    
    def get_params(self) -> Params:
        params = self.params
        return Params(
            max_words     = params.max_words,
            max_len       = params.max_len,
            test_size     = params.test_size,
            random_state  = params.random_state,
            batch_size    = params.batch_size,
            epochs        = params.epochs,
            embedding_dim = params.embedding_dim,
            lstm_units    = params.lstm_units,
            dropout_rate  = params.dropout_rate,
            learning_rate = params.learning_rate
        )

In [ ]:
import pickle
import numpy as np
import logging

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    LSTM,
    Bidirectional,
    Dense,
    Dropout,
    SpatialDropout1D
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)
from tensorflow.keras.optimizers import Adam

# from src.IMDBSentimentAnalysis.entity import ModelTrainerConfig, Params

logger = logging.getLogger(__name__)


class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig, params: Params):
        self.config = config
        self.params = params


    # ── Step 1: Load Transformed Data ────────────────────────────
    def load_data(self, train_path: str, test_path: str):
        try:
            train = np.load(train_path)
            test  = np.load(test_path)

            X_train, y_train = train['X'], train['y']
            X_test,  y_test  = test['X'],  test['y']

            logger.info(f"✅ Train loaded      : {X_train.shape}")
            logger.info(f"   Test loaded       : {X_test.shape}")

            return X_train, X_test, y_train, y_test

        except Exception as e:
            logger.error(f"❌ Data loading failed: {e}")
            raise e


    # ── Step 2: Build LSTM Model ──────────────────────────────────
    def build_model(self) -> Sequential:
        try:
            model = Sequential([

                # ✅Add input_shape here to fix "unbuilt" issue
                Embedding(
                    input_dim    = self.params.max_words,
                    output_dim   = self.params.embedding_dim,
                    input_shape  = (self.params.max_len,)      
                ),

                SpatialDropout1D(self.params.dropout_rate),

                Bidirectional(LSTM(
                    units             = self.params.lstm_units,
                    return_sequences  = True,
                    dropout           = self.params.dropout_rate,
                    recurrent_dropout = 0.2
                )),

                Bidirectional(LSTM(
                    units             = self.params.lstm_units // 2,
                    return_sequences  = False,
                    dropout           = self.params.dropout_rate,
                    recurrent_dropout = 0.2
                )),

                Dense(64, activation='relu'),
                Dropout(self.params.dropout_rate),

                Dense(1, activation='sigmoid')
            ])

            model.compile(
                optimizer = Adam(learning_rate=self.params.learning_rate),
                loss      = 'binary_crossentropy',
                metrics   = ['accuracy']
            )

            #  Explicitly build the model
            model.build(input_shape=(None, self.params.max_len))

            model.summary(print_fn=logger.info)
            logger.info(" Model built successfully")

            return model

        except Exception as e:
            logger.error(f"❌ Model build failed: {e}")
            raise e


    # ── Step 3: Callbacks ─────────────────────────────────────────
    def get_callbacks(self) -> list:
        early_stopping = EarlyStopping(
            monitor   = 'val_loss',
            patience  = 3,
            restore_best_weights = True,
            verbose   = 1
        )

        model_checkpoint = ModelCheckpoint(
            filepath  = self.config.model_path,
            monitor   = 'val_accuracy',
            save_best_only = True,
            verbose   = 1
        )

        reduce_lr = ReduceLROnPlateau(
            monitor   = 'val_loss',
            factor    = 0.2,
            patience  = 2,
            min_lr    = 1e-6,
            verbose   = 1
        )

        return [early_stopping, model_checkpoint, reduce_lr]


    # ── Step 4: Train Model ───────────────────────────────────────
    def train(self, model, X_train, y_train, X_test, y_test):
        try:
            logger.info("Training started...")

            history = model.fit(
                X_train, y_train,
                validation_data = (X_test, y_test),
                epochs          = self.params.epochs,
                batch_size      = self.params.batch_size,
                callbacks       = self.get_callbacks(),
                verbose         = 1
            )

            logger.info("Training completed")
            return history

        except Exception as e:
            logger.error(f"Training failed: {e}")
            raise e


    # ── Step 5: Save History ──────────────────────────────────────
    def save_history(self, history):
        try:
            with open(self.config.history_path, 'wb') as f:
                pickle.dump(history.history, f)

            logger.info(f"History saved     : {self.config.history_path}")

        except Exception as e:
            logger.error(f"❌ History save failed: {e}")
            raise e


    # ── Step 6: Evaluate ──────────────────────────────────────────
    def evaluate(self, model, X_test, y_test):
        try:
            loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

            logger.info(f"Test Loss         : {loss:.4f}")
            logger.info(f"Test Accuracy     : {accuracy * 100:.2f}%")

            return loss, accuracy

        except Exception as e:
            logger.error(f"Evaluation failed: {e}")
            raise e


    # ── Master Method ─────────────────────────────────────────────
    def run(self, train_path: str, test_path: str):
        X_train, X_test, y_train, y_test = self.load_data(train_path, test_path)
        model                            = self.build_model()
        history                          = self.train(model, X_train, y_train, X_test, y_test)
        self.save_history(history)
        self.evaluate(model, X_test, y_test)

In [ ]:
import logging
# from src.IMDBSentimentAnalysis.config.configuration import ConfigurationManager
# from src.IMDBSentimentAnalysis.components.model_trainer import ModelTrainer

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

STAGE_NAME = "Model Trainer Stage"


class ModelTrainerPipeline:
    def __init__(self):
        pass

    def run(self):
        config          = ConfigurationManager()
        trainer_cfg     = config.get_model_trainer_config()
        transform_cfg   = config.get_data_transformation_config()
        params          = config.get_params()

        trainer = ModelTrainer(
            config = trainer_cfg,
            params = params
        )

        trainer.run(
            train_path = transform_cfg.train_data_path,
            test_path  = transform_cfg.test_data_path
        )


if __name__ == "__main__":
    try:
        logger.info(f">>>>>> {STAGE_NAME} started <<<<<<")
        pipeline = ModelTrainerPipeline()
        pipeline.run()
        logger.info(f">>>>>> {STAGE_NAME} completed <<<<<<\n\nx==========x")

    except Exception as e:
        logger.exception(e)
        raise e